## Part 1: CNN Architecture

In [1]:
## Part 1: CNN Architecture

import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(42)
np.random.seed(42)

class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        
        # Conv2D with 16 filters, kernel size 3×3, stride 1, padding 1
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)
        # MaxPooling2D with kernel size 2×2, stride 2
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        # Conv2D with 32 filters, kernel size 3×3, stride 1, padding 1
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1)
        # Fully connected layer with 100 units (adjusted for 32×32 input)
        self.fc1 = nn.Linear(32 * 8 * 8, 100)
        # Fully connected layer with 10 units
        self.fc2 = nn.Linear(100, 10)
        
        self.relu = nn.ReLU()

    def forward(self, x):
        # Conv1 -> ReLU -> MaxPool
        x = self.pool(self.relu(self.conv1(x)))
        # Conv2 -> ReLU -> MaxPool
        x = self.pool(self.relu(self.conv2(x)))
        # Flatten
        x = x.view(x.size(0), -1)
        # FC1 -> ReLU
        x = self.relu(self.fc1(x))
        # FC2
        x = self.fc2(x)
        return x

# Test with 32×32 input
model = CNN()
sample_input = torch.randn(1, 3, 32, 32)
output = model(sample_input)
print(f"Input shape: {sample_input.shape}")
print(f"Output shape: {output.shape}")
print(model)

Input shape: torch.Size([1, 3, 32, 32])
Output shape: torch.Size([1, 10])
CNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=2048, out_features=100, bias=True)
  (fc2): Linear(in_features=100, out_features=10, bias=True)
  (relu): ReLU()
)


## Part 2: Model Deployment

In [2]:
## Part 2: Model Deployment

from tqdm import tqdm
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import torch.optim as optim

# Check which device is available
device = (
    torch.device("mps")
    if torch.backends.mps.is_available()
    else torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
)
print(f"Using device: {device}")

BATCH_SIZE = 32
EPOCHS = 5

# Prepare the Data
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

classes = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

model = CNN().to(device)

datalogs = []
# Train the model
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)

for epoch in range(EPOCHS):
    running_loss = 0.0
    running_correct, running_total = 0, 0
    model.train()
    train_loader_with_progress = tqdm(
        iterable=train_loader, ncols=120, desc=f"Epoch {epoch+1}/{EPOCHS}"
    )
    for batch_number, (inputs, labels) in enumerate(train_loader_with_progress):
        inputs = inputs.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        # log data for tracking
        running_correct += (predicted == labels).sum().item()
        running_total += labels.size(0)
        running_loss += loss.item()
        if batch_number % 100 == 99:
            train_loader_with_progress.set_postfix(
                {
                    "avg accuracy": f"{running_correct/running_total:.3f}",
                    "avg loss": f"{running_loss/(batch_number+1):.4f}",
                }
            )
            datalogs.append(
                {
                    "epoch": epoch + batch_number / len(train_loader),
                    "train_loss": running_loss / (batch_number + 1),
                    "train_accuracy": running_correct / running_total,
                }
            )
    datalogs.append(
        {
            "epoch": epoch + 1,
            "train_loss": running_loss / len(train_loader),
            "train_accuracy": running_correct / running_total,
        }
    )
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {running_loss/len(train_loader):.4f}, Accuracy: {running_correct/running_total:.3f}")

print("Finished Training")
torch.save(model.state_dict(), "app/cifar10_model.pth")
print("Model saved!")

Using device: mps


Epoch 1/5: 100%|███████████████████████████████| 1563/1563 [00:15<00:00, 98.96it/s, avg accuracy=0.423, avg loss=1.5952]


Epoch 1/5 - Loss: 1.5864, Accuracy: 0.427


Epoch 2/5: 100%|██████████████████████████████| 1563/1563 [00:12<00:00, 123.94it/s, avg accuracy=0.543, avg loss=1.2849]


Epoch 2/5 - Loss: 1.2832, Accuracy: 0.544


...

Epoch 3/5 - Loss: 1.1603, Accuracy: 0.591


Epoch 4/5: 100%|██████████████████████████████| 1563/1563 [00:14<00:00, 110.84it/s, avg accuracy=0.620, avg loss=1.0753]


Epoch 4/5 - Loss: 1.0750, Accuracy: 0.620


Epoch 5/5: 100%|██████████████████████████████| 1563/1563 [00:14<00:00, 111.09it/s, avg accuracy=0.646, avg loss=1.0021]

Epoch 5/5 - Loss: 1.0025, Accuracy: 0.646
Finished Training
Model saved!


In [3]:
# Evaluate on test set
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")

Test Accuracy: 63.18%


## Part 3: Questions

In [4]:
# Question 1: Given an input image of size 32×32×3 and a convolutional layer with 8 filters of size 5×5,
# stride 1, and no padding, what is the output size?

input_size = 32
kernel_size = 5
stride = 1
padding = 0
num_filters = 8

output_size = (input_size + 2 * padding - (kernel_size - 1) - 1) // stride + 1
print(f"Output size = ({input_size} + 2×{padding} - ({kernel_size}-1) - 1) / {stride} + 1 = {output_size}")
print(f"Output volume: {output_size} × {output_size} × {num_filters}")

Output size = (32 + 2×0 - (5-1) - 1) / 1 + 1 = 28
Output volume: 28 × 28 × 8


In [5]:
# Question 2: How does the output size change if padding is changed to “same”?

padding_same = (kernel_size - 1) // 2
output_size = (input_size + 2 * padding_same - (kernel_size - 1) - 1) // stride + 1
print(f"Padding needed for 'same': {padding_same}")
print(f"Output size = ({input_size} + 2×{padding_same} - ({kernel_size}-1) - 1) / {stride} + 1 = {output_size}")
print(f"Output volume: {output_size} × {output_size} × {num_filters}")

Padding needed for 'same': 2
Output size = (32 + 2×2 - (5-1) - 1) / 1 + 1 = 32
Output volume: 32 × 32 × 8


In [6]:
# Quetion 3: If you apply a 3×3 filter with stride 2 and no padding to a 64×64 input, what
# is the output spatial size?

input_size = 64
kernel_size = 3
stride = 2
padding = 0

output_size = (input_size + 2 * padding - (kernel_size - 1) - 1) // stride + 1
print(f"Output size = ({input_size} + 2×{padding} - ({kernel_size}-1) - 1) / {stride} + 1 = {output_size}")
print(f"Output spatial size: {output_size} × {output_size}")

Output size = (64 + 2×0 - (3-1) - 1) / 2 + 1 = 31
Output spatial size: 31 × 31


In [7]:
# Question 4: You apply a max-pooling layer of size 2×2 with stride 2 on a 16×16 feature
# map. What is the output size?

input_size = 16
kernel_size = 2
stride = 2
padding = 0

output_size = (input_size + 2 * padding - (kernel_size - 1) - 1) // stride + 1
print(f"Output size = ({input_size} + 2×{padding} - ({kernel_size}-1) - 1) / {stride} + 1 = {output_size}")
print(f"Output spatial size: {output_size} × {output_size}")

Output size = (16 + 2×0 - (2-1) - 1) / 2 + 1 = 8
Output spatial size: 8 × 8


In [8]:
# Question 5: An image of shape 128×128 is passed through two successive convolutional
# layers. Each uses a 3×3 kernel, stride 1, and ‘same’ padding. What is the output shape?

input_size = 128
kernel_size = 3
stride = 1
padding = (kernel_size - 1) // 2  # same padding

# First convolutional layer
output_size_1 = (input_size + 2 * padding - (kernel_size - 1) - 1) // stride + 1
print(f"After Conv Layer 1: ({input_size} + 2×{padding} - ({kernel_size}-1) - 1) / {stride} + 1 = {output_size_1}")

# Second convolutional layer
output_size_2 = (output_size_1 + 2 * padding - (kernel_size - 1) - 1) // stride + 1
print(f"After Conv Layer 2: ({output_size_1} + 2×{padding} - ({kernel_size}-1) - 1) / {stride} + 1 = {output_size_2}")

print(f"Output spatial size: {output_size_2} × {output_size_2}")

After Conv Layer 1: (128 + 2×1 - (3-1) - 1) / 1 + 1 = 128
After Conv Layer 2: (128 + 2×1 - (3-1) - 1) / 1 + 1 = 128
Output spatial size: 128 × 128


**Question 6: In the examples in class, before starting the training loop we ran: model.train().
What happens if you remove that line?**

In the `EnhancedCNN` architecture which uses both Dropout and Batch Normalization,
removing `model.train()` would result in:

- **Dropout** would be disabled — no neurons would be randomly dropped during 
training, causing the model to overfit to the training data
- **Batch Normalization** would use stored statistics instead of computing them 
from the current batch, leading to incorrect normalizations during training

Both of these behaviors are only correctly applied when the model is explicitly 
set to training mode with `model.train()`.